# 01 · Target Factory — the targets, the panel and the model input

**What this stage is for.** It builds the target of every client-month from the target source by
the rule in `data.target` (or takes an upstream column), joins the sources into one panel, applies
the eligibility steps and saves the **model input**: the one table every later stage reads
(`docs/components/dataset.md`, D-131). Saving it freezes the config; later changes go through the
decision audit log (`record_decision`, D-125). To work on another use case, change the config
path in the setup cell and rerun.

**What to look at.**
1. **The target summary** (step 1): rows per month; `null_share`, the targets unknown at the
   panel edge, for leavers and across gaps; `null_value_share`, the rows whose window held a null
   in the target column (if a null means 0, set `data.target.null_as`); and, for the observation
   window only, the mean target, the positive rows and the positive events. A rate that drifts
   across the window, or a null share above a few percent, is worth a question upstream before
   anything is modelled.
2. **The join, eligibility and contract printout** (step 2): the share of the spine's rows each
   source matched, the rows each `before: eda` filter dropped and each `before: model` filter
   tagged, the positive rate against `data.rate_band.max`, the null callout. A filter that drops
   nothing, or almost everything, usually means a wrong condition.
3. **The window frame** (step 3): what EDA will see, every client after the pre-EDA filters with
   the pre-model tags as columns, and the collect printout against the budget.

**Decisions to record:** none here. The data definition is the config itself: if the targets or
the eligibility are wrong, change the config and rerun; the data-definition hash changes with it.

**Docs:** `docs/components/target_factory.md` · `docs/components/dataset.md` ·
`docs/components/config.md` · the next stage: `02_eda` (M2.2).


In [ ]:
from pathlib import Path

import core_modelling
from core_modelling.config import load_config
from core_modelling.dataset import build_frames, save_model_input
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.target_factory import run_target_factory

# The use case and where the framework runs: absolute paths, never relative to the working
# directory (T5). The repo's example is found from the editable install; a use-case repo writes
# its own absolute config path here and reads the environment from $CORE_MODELLING_ENVIRONMENT.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"
ENTITY = "entity_a"
ENVIRONMENT = REPO / "conf/local/environment.yml"  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(str(ENVIRONMENT) if ENVIRONMENT else None)
spark = get_spark_session(env)
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.data.target, cfg.time, sep="\n")

## 1. The targets
The summary covers every row of the target source, before eligibility. Check the per-month rows
and null shares first, then the window months' rate: the `all` row pools the window.


In [ ]:
summary = run_target_factory(cfg, ENTITY, env, spark)

## 2. Save the model input
The panel is built (the join, eligibility and contract lines print as it goes) and written to the
path the environment file maps for `data.model_input`, with its metadata (the hashes, the
candidates, the logs) in the same folder. From here on the config is frozen.


In [ ]:
path = save_model_input(cfg, ENTITY, env, spark)

## 3. The window frame
What EDA will see, read back from the model input: every client after the pre-EDA filters, the
pre-model tags (here `non_holders`) and `eligible` as columns. The share tagged per month is the
eligibility funnel; the collect printout says how big the frame is against the budget.


In [ ]:
frames = build_frames(cfg, ENTITY, env, spark, stage="eda")
window = frames.window
print(
    len(window),
    "rows,",
    len(frames.meta["candidates"]),
    "candidate columns;",
    "lineage excluded:",
    frames.meta["lineage"],
)
window.groupby("month")[[*frames.meta["tags"], "eligible"]].mean()

In [ ]:
window.head()